# J1 — Des textes à une recherche utile

**NLP Avancé (BERT, GPT, Hugging Face) · M2 IA · Jour 1 · 240 min · Chrys NIONGOLO**

Travail en binôme : alterner la personne qui conduit et celle qui explique. Enregistrer une copie dans Drive, puis conserver le notebook exécuté et le JSON final.

## À la fin, vous saurez

- Expliquer ce que représente un token et inspecter les cas limites français.
- Construire une recherche TF-IDF, puis la comparer à des embeddings.
- Mesurer Recall@k et MRR, et justifier une erreur avec un exemple.

## Parcours de l’après-midi

| Étape | Minutes |
|---|---:|
| Cadrage, données et tokenisation | 35 |
| TF-IDF et recherche | 50 |
| Tokenizer BPE et expériences | 40 |
| Embeddings et comparaison | 55 |
| Analyse d’erreurs, test et restitution | 60 |

Les durées sont des budgets d’activité incluant lecture, discussion, essais et débrief ; ce ne sont pas des durées de calcul promises. Les corpus intégrés sont originaux, fictifs et minuscules. Ils servent à comprendre un protocole, pas à certifier une qualité métier.

**Règle d’expérience :** écrire une prédiction avant de lancer une variante ; changer un seul facteur ; choisir sur la validation ; ouvrir le test après ce choix.


## 0. Préparer Colab

Importer ce fichier dans https://colab.research.google.com/ via Fichier → Importer.
Choisir Exécution → Modifier le type d’exécution → Python 3 → GPU T4 si disponible.
La disponibilité du T4 dépend des quotas Colab. Aucun compte Hugging Face n’est requis
pour les modèles publics utilisés ici. Une connexion internet est nécessaire pour les
bibliothèques et les poids ; les données pédagogiques sont déjà dans le notebook.

Exécuter la cellule suivante **avant les autres imports**. Si elle demande un redémarrage,
choisir Exécution → Redémarrer la session puis relancer depuis le début. Ne pas réinstaller
un autre PyTorch dans un runtime CUDA fonctionnel. Les versions sont volontairement fixées.
Aucun entraînement n’est lancé par cette cellule. Aucun envoi au Hub n’est automatique.

**Runpod L4 facultatif :** ce notebook fonctionne dans Jupyter avec Python 3.10+
et PyTorch CUDA préinstallé. Les chemins sont relatifs ; aucun import Google Colab
n’est requis. Arrêter le Pod après sauvegarde des résultats pour interrompre sa facturation.


In [ ]:
PINNED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.6', 'scikit-learn': '1.7.2', 'transformers': '4.56.2', 'tokenizers': '0.22.0', 'huggingface-hub': '0.35.3', 'accelerate': '1.10.1', 'sentence-transformers': '5.1.1'}
import sys, subprocess, importlib.metadata as metadata
assert sys.version_info >= (3, 10), "Utiliser Python 3.10 ou ultérieur."
module_names = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",
                "huggingface-hub": "huggingface_hub", "rouge-score": "rouge_score"}
to_install = []
loaded_mismatch = []
for package, version in PINNED.items():
    module = sys.modules.get(module_names.get(package, package))
    if module is not None and getattr(module, "__version__", version) != version:
        loaded_mismatch.append(package)
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != version:
        to_install.append(f"{package}=={version}")
if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
if loaded_mismatch:
    raise RuntimeError("Redémarrer la session Colab, puis relancer depuis le début : "
                       + ", ".join(loaded_mismatch))
print("Versions de référence installées. Continuer avec la cellule suivante.")


In [ ]:
import os, json, time, random, platform, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
versions = {p: metadata.version(p) for p in PINNED}
versions["python"] = platform.python_version()
results = {"seed": SEED, "versions": versions, "validation": "exécuté par le binôme"}
OUTPUT = Path("resultats")
OUTPUT.mkdir(exist_ok=True)
display(pd.Series(versions, name="version effective"))


## 1. Partir d’une question concrète

Notre équipe reçoit des questions clients et possède une petite base de réponses.
Le premier objectif n’est pas de générer du texte : il faut retrouver la bonne fiche.
Avant le calcul, associer oralement les trois premières requêtes à une fiche.

**Mini-exercice (5 min).** Que faut-il conserver dans « Je ne veux plus annuler » ?
Quels éléments une séparation par espace traite-t-elle mal dans « l’appli », « 15,50 € » et « ré-expédition » ?


In [ ]:
import re
faq = [
    ("suivi", "Suivre un colis", "Consultez le numéro de suivi pour localiser votre colis et connaître les étapes du transport."),
    ("non_recu", "Colis indiqué livré", "Le colis est marqué livré mais absent : vérifiez le lieu de dépôt puis contactez le support."),
    ("facture", "Télécharger une facture", "Retrouvez la facture et le justificatif comptable dans le détail de votre commande."),
    ("double", "Double débit", "Deux paiements identiques : vérifiez les autorisations bancaires puis signalez un éventuel double débit."),
    ("mot_passe", "Retrouver son accès", "Utilisez le lien de réinitialisation si votre mot de passe est oublié."),
    ("email", "Changer le courriel", "L'adresse email du compte peut être modifiée depuis les paramètres du profil."),
    ("retour", "Renvoyer un article", "Ouvrez une demande de retour et consultez les conditions avant de renvoyer le produit."),
    ("etiquette", "Bordereau de retour", "Téléchargez l'étiquette de retour depuis votre demande ; le support aide en cas d'échec."),
    ("bug", "Application qui plante", "Mettez à jour l'application puis signalez le plantage, votre appareil et les étapes de reproduction."),
    ("image", "Photos absentes", "Si les images des produits ne chargent pas, vérifiez la connexion puis actualisez le navigateur."),
    ("annuler", "Annuler une commande", "Demandez l'annulation au support avant l'expédition. Une commande expédiée suit la procédure de retour."),
    ("conserver", "Conserver une commande", "Pour conserver votre achat, ne demandez pas son annulation. Contactez le support si une demande est déjà en cours."),
]
docs = pd.DataFrame(faq, columns=["id", "titre", "texte"])
dev_queries = [
    ("Le colis n'avance plus dans le suivi", "suivi"),
    ("Il me faut une pièce pour ma comptabilité", "facture"),
    ("Ma banque affiche deux prélèvements", "double"),
    ("J'ai perdu mon secret de connexion", "mot_passe"),
    ("L'appli se ferme brutalement", "bug"),
    ("Je ne veux pas annuler mon achat", "conserver"),
    ("Le livreur dit livré mais je n'ai rien", "non_recu"),
    ("Comment imprimer le document de renvoi ?", "etiquette"),
]
# Ne pas utiliser cette liste pour ajuster les paramètres.
final_queries = [
    ("J'ai besoin du reçu de mon achat", "facture"),
    ("Les visuels sont tous invisibles", "image"),
    ("Je souhaite utiliser une autre adresse électronique", "email"),
    ("Je vais vous restituer cet article", "retour"),
    ("Je souhaite stopper ma commande avant son départ", "annuler"),
    ("J'aimerais garder cette commande finalement", "conserver"),
]
display(docs)
exemple = "Je ne veux plus annuler l'appli à 15,50 € : ré-expédition ?"
print("Espace :", exemple.split())
print("Expression régulière :", re.findall(r"\w+|[^\w\s]", exemple))


In [ ]:
reponses = {"negation": "À compléter", "unite_token": "À compléter", "hypothese_semantique": "À compléter"}
# Remplacer les réponses par vos observations, sans attendre de connaître les scores.
print(reponses)


### Repères formateur — éléments de correction

La négation change l’action demandée. Supprimer « ne » et « pas » comme stop words peut rapprocher des intentions opposées. Un token est une unité du tokenizer : ni nécessairement un mot ni nécessairement une syllabe. Une bonne tokenisation ne garantit pas une bonne compréhension.


## 2. Une baseline que l’on comprend : TF-IDF

TF compte les termes du document ; IDF réduit le poids des termes présents presque partout.
Deux vecteurs qui pointent dans une direction proche ont un cosinus élevé. Cette proximité
est un signal de recherche ; ce n’est pas une probabilité que la réponse soit correcte.

**Avant exécution.** Prédire si « pièce pour ma comptabilité » retrouvera la fiche facture.
Après exécution, lire les mots les plus pondérés et les trois premières fiches.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
NGRAM_RANGE = (1, 1)
vectorizer = TfidfVectorizer(ngram_range=NGRAM_RANGE, strip_accents="unicode")
document_vectors = vectorizer.fit_transform(docs.texte)
def lexical_scores(queries):
    return cosine_similarity(vectorizer.transform(queries), document_vectors)
terms = vectorizer.get_feature_names_out()
weights = document_vectors[2].toarray().ravel()
display(pd.DataFrame({"terme": terms, "poids": weights}).sort_values("poids", ascending=False).head(8))
query = "Il me faut une pièce pour ma comptabilité"
ranking = np.argsort(-lexical_scores([query])[0])[:3]
display(docs.iloc[ranking].assign(score=lexical_scores([query])[0][ranking]))
print("Dimensions documents × vocabulaire :", document_vectors.shape)


### Mesurer une recherche

Recall@3 vaut 1 si la bonne fiche figure dans les trois premiers résultats, sinon 0.
Le rang réciproque vaut 1/rang de la bonne fiche ; MRR en fait la moyenne.
Ici chaque question a une seule fiche attendue. Des données réelles peuvent accepter plusieurs réponses.

**Expérience A (15 min).** Comparer unigrammes et unigrammes+bigrams sur la validation.
Consigner le changement et garder les mêmes requêtes. Ne modifier ni les fiches ni les labels pour améliorer le score.


In [ ]:
def retrieval_metrics(scores, queries):
    expected = [docs.index[docs.id == expected_id][0] for _, expected_id in queries]
    order = np.argsort(-scores, axis=1, kind="stable")
    ranks = np.array([np.flatnonzero(order[i] == target)[0] + 1 for i, target in enumerate(expected)])
    return {"recall_at_1": float(np.mean(ranks <= 1)), "recall_at_3": float(np.mean(ranks <= 3)),
            "mrr": float(np.mean(1 / ranks)), "n": len(queries)}
dev_scores_lexical = lexical_scores([q for q, _ in dev_queries])
metrics_lexical = retrieval_metrics(dev_scores_lexical, dev_queries)
print(metrics_lexical)
experiments = [{"method": "tfidf", "ngram_range": NGRAM_RANGE, **metrics_lexical}]
assert retrieval_metrics(np.eye(len(docs))[:2], [("q1", docs.id.iloc[0]), ("q2", docs.id.iloc[1])])["mrr"] == 1.0


### Repères formateur — éléments de correction

Les bigrammes peuvent préserver des expressions mais accroissent la rareté. Il n’existe pas de gagnant imposé. Un score ex aequo sur une matrice de zéros dépend ici de l’ordre stable des fiches : faire repérer ce cas. Le petit test d’identité vérifie le calcul de la métrique, pas la qualité du moteur.


## 3. Construire un petit tokenizer BPE

BPE apprend à regrouper des symboles fréquents. Augmenter le vocabulaire produit souvent
moins de morceaux par texte, mais demande plus de paramètres d’embeddings dans un modèle.
Nous n’entraînons ici qu’un tokenizer, sans réseau de neurones.

**Expérience B (25 min).** Comparer des vocabulaires de 120 et 240 unités. Tester une négation,
un nom jamais rencontré et une faute. Mesurer le nombre de tokens et examiner les unités inconnues.


In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
def train_bpe(vocab_size):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator(docs.texte.tolist(), trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=1, special_tokens=["[UNK]", "[PAD]"]))
    return tok
phrases = ["Je ne veux pas annuler", "réinitialisation du mot de passe", "N'Djamena", "coliiiiis 😅"]
bpe_rows = []
for size in [120, 240]:
    tok = train_bpe(size)
    for phrase in phrases:
        encoded = tok.encode(phrase)
        bpe_rows.append({"vocab": tok.get_vocab_size(), "phrase": phrase,
                         "tokens": encoded.tokens, "nombre": len(encoded.ids),
                         "inconnus": encoded.tokens.count("[UNK]")})
display(pd.DataFrame(bpe_rows))
# Ce tokenizer miniature ne remplace jamais celui d'un modèle préentraîné.


### Repères formateur — éléments de correction

Ce BPE minuscule illustre les fusions, pas une couverture réaliste du français. Des caractères absents du corpus deviennent inconnus. Le vocabulaire réellement obtenu peut être inférieur à la cible. Un tokenizer byte-level limiterait ce problème de couverture, au prix d’autres arbitrages. Changer le tokenizer d’un modèle préentraîné sans réapprendre ses embeddings casse la correspondance identifiant-vecteur.


## 4. Passer aux embeddings de phrases

Un encodeur déjà entraîné peut rapprocher des formulations différentes. Il a appris sur d’autres
données ; notre appel encode ne modifie aucun poids. Nous normalisons les vecteurs : leur produit
scalaire devient alors le cosinus. Prédire une requête qui devrait s’améliorer et une qui risque d’échouer.

Si le téléchargement est impossible, passer USE_PRETRAINED à False et terminer l’analyse TF-IDF.
Noter l’étape non exécutée au lieu de recopier un résultat d’un autre groupe.


In [ ]:
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
USE_PRETRAINED = True
embedder = None
metrics_semantic = None
if USE_PRETRAINED:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = embedder.encode(docs.texte.tolist(), normalize_embeddings=True)
    def semantic_scores(queries):
        q = embedder.encode(queries, normalize_embeddings=True)
        return q @ embeddings.T
    dev_scores_semantic = semantic_scores([q for q, _ in dev_queries])
    metrics_semantic = retrieval_metrics(dev_scores_semantic, dev_queries)
    print("Dimensions :", embeddings.shape, "; métriques :", metrics_semantic)
    assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5)
else:
    print("Embeddings préentraînés non exécutés.")


In [ ]:
analysis = []
for i, (query, expected) in enumerate(dev_queries):
    row = {"question": query, "attendu": expected,
           "tfidf": docs.id.iloc[int(dev_scores_lexical[i].argmax())]}
    if embedder is not None:
        row["embedding"] = docs.id.iloc[int(dev_scores_semantic[i].argmax())]
    analysis.append(row)
display(pd.DataFrame(analysis))
if embedder is not None:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=SEED).fit_transform(embeddings)
    plt.figure(figsize=(9, 5))
    plt.scatter(xy[:, 0], xy[:, 1])
    for i, name in enumerate(docs.id):
        plt.annotate(name, xy[i], fontsize=9)
    plt.title("Projection PCA : vue partielle, les distances originales ne sont pas préservées")
    plt.show()


## 5. Décider et expliquer — puis ouvrir le test

Choisir la méthode sur les résultats de validation, puis renseigner CHOSEN_METHOD.
Examiner deux erreurs : manque lexical, négation, ambiguïté ou mauvaise fiche de référence ?
Une visualisation en deux dimensions ne constitue pas une preuve de performance.

**Livrable (30 min).** Tableau validation/test, deux erreurs expliquées, un exemple où la baseline
suffit, un exemple où une annotation doit être discutée. Proposer un protocole sur 100 questions réelles
anonymisées sans supposer qu’on possède déjà leur autorisation d’usage.


In [ ]:
CHOSEN_METHOD = "tfidf"  # Choisir après validation : "tfidf" ou "embedding".
assert CHOSEN_METHOD in {"tfidf", "embedding"}
if CHOSEN_METHOD == "embedding" and embedder is None:
    raise ValueError("Choisir une méthode effectivement exécutée.")
final_scores = (semantic_scores if CHOSEN_METHOD == "embedding" else lexical_scores)([q for q, _ in final_queries])
final_metrics = retrieval_metrics(final_scores, final_queries)
print("Test final, sans retouche après lecture :", final_metrics)
results.update({"validation_tfidf": metrics_lexical, "validation_embedding": metrics_semantic,
                "chosen_method": CHOSEN_METHOD, "test": final_metrics, "reponses": reponses,
                "bpe": bpe_rows})
conclusion_binome = "À compléter : résultat observé, deux erreurs, limite de généralisation."


### Repères formateur — éléments de correction

Attendre une décision argumentée, pas nécessairement le choix de l’encodeur. Avec six questions de test, une erreur change Recall@1 d’environ 0,167. Un cas négatif doit déclencher une discussion de l’intention. Demander pourquoi ajouter le test au corpus d’apprentissage du tokenizer ou modifier les fiches après son inspection invaliderait le protocole de comparaison initial.


## Ressources pour prolonger

- [HF — Tokenizers](https://huggingface.co/learn/llm-course/fr/chapter6/1)
- [Modèle multilingue et limites](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
- [Scikit-learn — extraction de caractéristiques textuelles](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)

Liens et API consultés le 4 octobre 2026. Les exercices et données de ce notebook sont originaux. Les téléchargements de modèles et l’exécution Colab doivent être vérifiés avant la séance.


## Trace à rendre

Compléter vos réponses et ajouter une observation appuyée par une sortie réelle.
Exécuter la dernière cellule, télécharger le JSON dans le panneau Fichiers de Colab,
puis télécharger votre notebook avec ses sorties. Les fichiers du runtime sont temporaires.
Ne pas remplacer une mesure absente par un chiffre plausible. Indiquer les étapes non exécutées.


In [ ]:
results["notebook"] = '01_j1_textes_recherche'
results["conclusion_binome"] = globals().get("conclusion_binome", "À compléter")

result_file = OUTPUT / "j1_resultats.json"
result_file.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Fichier à télécharger :", result_file.resolve())
